# 03 · Banderas rojas, score de riesgo y data mart · ORCA v1.0

**Entrada:** `data/curated/DS-01_contratos_curated.parquet` (quality gate = PUBLICAR) + `data/raw/DS-02_*.parquet` (solo columnas de competencia, sin PII).
**Salida:** zona `data/mart/` — lo único que lee el dashboard.

Responde las preguntas de negocio del documento 01 §3 con **9 banderas rojas versionadas** (`notebooks/reglas_riesgo.json`).
El score **prioriza** la revisión: *la máquina mide, la persona decide*. Ninguna bandera prueba una irregularidad.

In [1]:
from pathlib import Path
import json, sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks"))
pd.set_option("display.max_columns", 50, "display.width", 200, "display.max_colwidth", 80)
CURATED, MART, RAW = ROOT / "data" / "curated", ROOT / "data" / "mart", ROOT / "data" / "raw"
GOB = ROOT / "documentacion" / "gobierno"
GOB.mkdir(parents=True, exist_ok=True)

## 1. Reglas versionadas (AD-4 de la arquitectura)

Cada score queda asociado a la versión de las reglas. Cambiar un peso o umbral exige subir la versión: así el Data Steward recalibra con los falsos positivos (mejora M6) sin tocar el código.

In [2]:
import riesgo
reglas = riesgo.cargar_reglas()
print("Versión:", reglas["version"], "·", reglas["aprobada_por"])
pd.DataFrame([{"código": k, "bandera": v["nombre"], "pregunta": v["pregunta"], "peso": v["peso"], "regla": v["regla"]}
              for k, v in reglas["banderas"].items()])

Versión: 1.0.0 · Data Owner (Luis Ángel Seoanes Ovieso) · propuesta del Data Steward (Julian Amariles Taborda)


,código,bandera,pregunta,peso,regla
0,R1,Contratación directa,P1,10,La modalidad empieza por 'Contratación directa' (con o sin ofertas).
1,R2,Firma directa en ventana atípica,P2,15,Contratación directa firmada en una ventana atípica: el mes previo a la rest...
2,R3,Concentración entidad–proveedor,P3,15,El par entidad–proveedor tiene al menos 3 contratos y concentra ≥ 20 % del v...
3,R4,Persona natural en varias entidades a la vez,P4,10,El contratista es persona natural y en algún mes de ejecución del contrato t...
4,R5,Valor atípico para su tipo y modalidad,P5,15,"log10(valor) supera Q3 + 1,5·IQR de su grupo tipo de contrato × modalidad (g..."
5,R6,Proceso competitivo con un único oferente,P6,15,"Modalidad competitiva (licitación, selección abreviada, concurso, mínima cua..."
6,R7,Urgencia manifiesta,P1,10,La causal de la modalidad menciona urgencia manifiesta (contratación directa...
7,R8,Prórroga desproporcionada,P5,5,Días adicionados ≥ 60 y ≥ 50 % del plazo pactado (inicio→fin).
8,R9,Inconsistencia del dato,Calidad,5,"Valor $0 sin causal de valor cero, o fecha de fin anterior a la de inicio. S..."


## 2. Ejecución curated → mart

El motor verifica primero el quality gate del lote (política: solo se publica lo que pasó el gate) y al final ejecuta los controles de gobierno (nomenclatura y PII en claro = 0). Si un control falla, el mart **no** se escribe.

In [3]:
man = riesgo.procesar(str(CURATED), str(RAW / "DS-02_*.parquet"), str(MART))
print("Corte:", man["corte"], "· gate:", man["entradas"]["curated"]["decision_quality_gate"],
      f"({man['entradas']['curated']['puntaje_calidad']} %)")
print("Contratos con su proceso en DS-02:", man["entradas"]["procesos_DS-02"]["pct_contratos_con_proceso_en_DS02"], "%")
pd.Series(man["controles_gobierno"], name="control")

Corte: 2026-11-05 · gate: PUBLICAR (99.66 %)
Contratos con su proceso en DS-02: 99.13 %


columnas_fuera_de_nomenclatura                 []
columnas_pii_prohibidas                        []
personas_naturales_con_documento_o_nombre       0
seudonimos_validos                           True
pii_en_claro_mart                               0
Name: control, dtype: object

In [4]:
c = pd.read_parquet(MART / "mart_contratos_riesgo.parquet")
e = pd.read_parquet(MART / "mart_entidades.parquet")
p = pd.read_parquet(MART / "mart_pares_entidad_proveedor.parquet")
bcols = [x for x in c.columns if x.startswith("ind_r")]
banderas = pd.DataFrame({"contratos": c[bcols].sum(), "pct": (100 * c[bcols].mean()).round(2),
                         "valor_COP": [c.loc[c[b], "val_contrato"].sum() for b in bcols]})
banderas

,contratos,pct,valor_COP
ind_r1_contratacion_directa,248578,84.43,3.958474e+13
ind_r2_ventana_atipica,47352,16.08,5.607677e+12
ind_r3_concentracion_par,2993,1.02,6.017405e+12
ind_r4_simultaneidad_pn,4389,1.49,3.247996e+11
ind_r5_valor_atipico,4294,1.46,1.988480e+13
ind_r6_unico_oferente,16276,5.53,1.803538e+13
ind_r7_urgencia_manifiesta,362,0.12,3.170947e+11
ind_r8_prorroga_desproporcionada,2658,0.90,2.044362e+12
ind_r9_inconsistencia_dato,2433,0.83,5.462004e+09


In [5]:
niveles = c.groupby("niv_riesgo").agg(contratos=("id_contrato", "size"), valor=("val_contrato", "sum"))
niveles["pct_contratos"] = (100 * niveles["contratos"] / len(c)).round(2)
niveles.reindex(["Alto", "Medio", "Bajo", "Sin señales"])

,contratos,valor,pct_contratos
niv_riesgo,,,
Alto,2262,1.016746e+13,0.77
Medio,55385,1.942717e+13,18.81
Bajo,204432,1.914142e+13,69.44
Sin señales,32330,9.720562e+12,10.98


## 3. Respuestas a las preguntas de negocio

### P1 · Entidades que contratan directo por encima de su propia historia

In [6]:
p1 = e[e["ind_p1_directa_sobre_historia"] & e["ind_sujeto_vigilancia"]].sort_values("num_delta_directa_pp", ascending=False)
print(len(p1), "entidades territoriales con ≥ 10 pp más de contratación directa en 2026 que en 2024–2025")
p1[["nom_entidad", "nom_municipio", "cant_contratos_2024_2025", "cant_contratos_2026",
    "pct_directa_2024_2025", "pct_directa_2026", "num_delta_directa_pp"]].head(10).round(3)

12 entidades territoriales con ≥ 10 pp más de contratación directa en 2026 que en 2024–2025


,nom_entidad,nom_municipio,cant_contratos_2024_2025,cant_contratos_2026,pct_directa_2024_2025,pct_directa_2026,num_delta_directa_pp
159,MUNICIPIO DE ZARAGOZA,ZARAGOZA,121.0,427.0,0.322,0.745,42.242
15,INSTITUCIÓN UNIVERSITARIA PÚBLICA DE BELLO - IUPBELLO,BELLO,80.0,41.0,0.538,0.927,38.933
0,ALCALDIA MUNICIPIO DE NECOCLI,NECOCLÍ,101.0,307.0,0.703,0.883,17.977
6,INSTITUTO MUNICIPAL DE DEPORTES Y RECREACIÓN DE APARTADÓ,APARTADÓ,173.0,163.0,0.763,0.933,16.951
10,MUNICIPIO DE EL BAGRE,EL BAGRE,453.0,392.0,0.620,0.788,16.796
2,PERSONERIA MUNICIPAL DE BELLO,BELLO,61.0,39.0,0.836,1.000,16.393
1,MUNICIPIO DE BRICEÑO,BRICEÑO,165.0,236.0,0.642,0.805,16.266
12,ALCALDIA MUNICIPIO DE SONSON,SONSÓN,1144.0,542.0,0.674,0.836,16.184
162,PROVINCIA ADMINISTRATIVA Y DE PLANIFICACIÓN CARTAMA,MEDELLÍN,69.0,59.0,0.797,0.949,15.205
154,MUNICIPIO DE MACEO,MACEO,346.0,214.0,0.497,0.626,12.906


### P2 · Ventanas atípicas (antes de la Ley de Garantías y fin de vigencia)

In [7]:
r2 = c[c["ind_r2_ventana_atipica"]]
print(f"{len(r2):,} contratos directos firmados en ventana atípica · ${r2['val_contrato'].sum()/1e12:,.2f} billones")
r2.groupby(r2["fec_firma"].dt.to_period("M")).size().rename("contratos")

47,352 contratos directos firmados en ventana atípica · $5.61 billones


fec_firma
2024-12      828
2025-12      639
2026-01    45885
Freq: M, Name: contratos, dtype: int64

### P3 · Concentración entidad–proveedor

In [8]:
p3 = p[p["ind_r3_concentracion_par"]].sort_values("pct_valor_entidad", ascending=False)
print(len(p3), "pares con concentración")
p3[["nom_entidad", "nom_proveedor", "cant_contratos", "val_total", "pct_valor_entidad", "pct_contratos_entidad"]].head(10).round(3)

116 pares con concentración


,nom_entidad,nom_proveedor,cant_contratos,val_total,pct_valor_entidad,pct_contratos_entidad
174,ASOCIACIÓN CANAL LOCAL DE TELEVISIÓN DE MEDELLIN - TELEMEDELLIN,EMPRESA DE SERVICIOS TEMPORALES MAXEMPLEOS S.A.S.,3,31645000000.0,0.868,0.045
106,MUNICIPIO DE PUERTO NARE,Persona natural · 93d53ca8,8,49265935700.0,0.679,0.009
1,INSTITUTO DE DEPORTES Y RECREACION DE MEDELLIN,EMPRESA DE DESARROLLO URBANO DE MEDELLIN,9,1879145421497.0,0.665,0.001
749,INSTITUTO MUNICIPAL DE DEPORTES DEL MUNICIPIO DE TURBO,COPORACION EDUCATIVA URABA SIN LIMITES,19,5904303272.0,0.645,0.073
3563,ESTABLECIMIENTO PENITENCIARIO MEDIANA SEGURIDAD Y CARCELARIO SONSON,INVERSIONES DOGICA S.A.S,11,361338029.0,0.642,0.333
402,LA PROVINCIA ADMINISTRATIVA Y DE PLANIFICACION DEL AGUA BOSQUES Y EL TURISMO,La Provincia de La Paz,3,12146566024.0,0.625,0.017
14,AGENCIA DE EDUCACIÓN POSTSECUNDARIA DE MEDELLÍN -SAPIENCIA-,INSTITUTO PARA EL DESARROLLO DE ANTIOQUIA IDEA,5,383202180502.0,0.614,0.003
192,MUNICIPIO DE ANDES,EMPRESA DE DESARROLLO URBANO DE ANDES - EDUAN,57,27511365689.540001,0.563,0.099
134,ASOCIACIÓN DE MUNICIPIOS DEL NORTE ANTIOQUEÑO- AMUNORTE,CORPORACION AFROPROGRESO DIEGO LUIS CORDOBA,160,40977369005.599998,0.554,0.519
90,INSTITUTO MUNICIPAL DE CULTURA; RECREACIÓN Y DEPORTE DE ITAGUI,Coldeporte,5,57446907272.0,0.51,0.051


### P4 · Personas naturales con contratos simultáneos en varias entidades

Se compara por **seudónimo**: el analista nunca ve la cédula, y aun así la señal se puede calcular.

In [9]:
p4 = c[c["ind_r4_simultaneidad_pn"]]
print(f"{len(p4):,} contratos · {p4['id_proveedor_seud'].nunique():,} contratistas persona natural")
p4["cant_entidades_simultaneas"].value_counts().sort_index().rename("contratos por máximo de entidades simultáneas")

4,389 contratos · 523 contratistas persona natural


cant_entidades_simultaneas
3     2452
4      780
5      418
6      389
7      143
8       64
9       14
10      53
11      20
12      24
13      32
Name: contratos por máximo de entidades simultáneas, dtype: int64

### P5 · Valores atípicos para su tipo y modalidad

In [10]:
p5 = c[c["ind_r5_valor_atipico"]].assign(veces=lambda x: x["val_contrato"] / x["val_limite_atipico_grupo"])
print(f"{len(p5):,} contratos atípicos para su grupo")
p5.nlargest(10, "val_contrato")[["id_contrato", "nom_entidad", "cod_tipo_contrato", "cod_modalidad", "val_contrato", "veces"]].round(1)

4,294 contratos atípicos para su grupo


,id_contrato,nom_entidad,cod_tipo_contrato,cod_modalidad,val_contrato,veces
97884,CO1.PCCNTR.9953923,INSTITUTO DE DEPORTES Y RECREACION DE MEDELLIN,Otro,Contratación directa,1.159398e+12,38.1
5448,CO1.PCCNTR.7171517,DEPARTAMENTO DE ANTIOQUIA,Obra,Contratación Directa (con ofertas),7.125692e+11,14.1
205991,CO1.PCCNTR.7969114,DEPARTAMENTO DE ANTIOQUIA,Obra,Contratación Directa (con ofertas),5.522739e+11,10.9
41916,CO1.PCCNTR.8258264,DISTRITO ESPECIAL DE CIENCIA TECNOLOGIA E INNOVACION DE MEDELLIN,Otro,Contratación Directa (con ofertas),4.437244e+11,4.6
88998,CO1.PCCNTR.7862082,DEPARTAMENTO DE ANTIOQUIA,Otro,Contratación directa,3.580161e+11,11.8
55783,CO1.PCCNTR.7250064,DISTRITO ESPECIAL DE CIENCIA TECNOLOGIA E INNOVACION DE MEDELLIN,Otro,Contratación Directa (con ofertas),3.374778e+11,3.5
292990,CO1.PCCNTR.6451319,SUBCUENTA DEL SISTEMA DE PARQUES NACIONALES NATURALES DEL FONDO NACIONAL AMB...,Otro,Contratación Directa (con ofertas),3.300000e+11,3.4
91609,CO1.PCCNTR.8364608,INSTITUTO DE DEPORTES Y RECREACION DE MEDELLIN,Otro,Contratación directa,2.486861e+11,8.2
39683,CO1.PCCNTR.8034294,INSTITUTO DE DEPORTES Y RECREACION DE MEDELLIN,Otro,Contratación directa,2.359484e+11,7.8
247596,CO1.PCCNTR.5945659,ICBF REGIONAL ANTIOQUIA,Otro,Contratación directa,2.274461e+11,7.5


### P6 · Procesos competitivos con un único oferente (cruce con DS-02)

In [11]:
p6 = c[c["ind_r6_unico_oferente"]]
print(f"{len(p6):,} contratos de procesos competitivos con una sola oferta · ${p6['val_contrato'].sum()/1e9:,.0f} mil millones")
p6["cod_modalidad"].value_counts()

16,276 contratos de procesos competitivos con una sola oferta · $18,035 mil millones

cod_modalidad
Mínima cuantía                                                 6805
Contratación Directa (con ofertas)                             5813
Contratación régimen especial (con ofertas)                    1402
Selección Abreviada de Menor Cuantía                           1081
Selección abreviada subasta inversa                             840
Licitación pública                                              217
Concurso de méritos abierto                                      82
Seleccion Abreviada Menor Cuantia Sin Manifestacion Interes      19
Licitación pública Obra Publica                                  11
Enajenación de bienes con subasta                                 4
Enajenación de bienes con sobre cerrado                           2
Name: count, dtype: Int64

## 4. Priorización de entidades (insumo del Plan de Vigilancia)

In [12]:
e[e["ind_sujeto_vigilancia"]].head(15)[["cant_ranking", "nom_entidad", "nom_municipio", "cant_contratos", "cant_alto",
                                           "val_riesgo_alto", "num_score_medio", "ind_p1_directa_sobre_historia", "num_indice_prioridad"]].round(2)

,cant_ranking,nom_entidad,nom_municipio,cant_contratos,cant_alto,val_riesgo_alto,num_score_medio,ind_p1_directa_sobre_historia,num_indice_prioridad
0,1,ALCALDIA MUNICIPIO DE NECOCLI,NECOCLÍ,408,8,2.724124e+10,15.98,True,73.3
1,2,MUNICIPIO DE BRICEÑO,BRICEÑO,401,1,9.606190e+09,14.68,True,70.3
2,3,PERSONERIA MUNICIPAL DE BELLO,BELLO,100,3,2.117727e+09,14.45,True,67.9
5,6,ALCALDIA MUNICIPIO DE ANORI,Sin municipio,544,2,2.001213e+08,13.52,True,63.4
6,7,INSTITUTO MUNICIPAL DE DEPORTES Y RECREACIÓN DE APARTADÓ,APARTADÓ,336,1,2.075640e+08,13.41,True,63.3
9,10,MUNICIPIO DE ITAGUI,ITAGÜÍ,944,118,6.076070e+11,20.06,False,62.7
10,11,MUNICIPIO DE EL BAGRE,EL BAGRE,845,1,2.500000e+08,11.82,True,61.6
12,13,ALCALDIA MUNICIPIO DE SONSON,SONSÓN,1686,1,2.300000e+08,10.96,True,60.5
13,14,DISTRITO ESPECIAL DE CIENCIA TECNOLOGIA E INNOVACION DE MEDELLIN,MEDELLÍN,6335,204,2.527707e+12,16.14,False,59.9
14,15,ASOCIACIÓN DE MUNICIPIOS DEL NORTE ANTIOQUEÑO- AMUNORTE,MEDELLÍN,308,28,3.369231e+10,20.28,False,58.9


## 5. Bandeja de alertas y conclusión

La bandeja de riesgo alto es la que recibe el equipo auditor (automatización `notificar_alertas.py`). Es una fracción pequeña del universo: eso es lo que hace viable revisarla con la misma planta de auditores.

In [13]:
alto = c[(c["niv_riesgo"] == "Alto") & c["ind_sujeto_vigilancia"]]
print(f"Bandeja: {len(alto):,} contratos ({100*len(alto)/c['ind_sujeto_vigilancia'].sum():.2f} % del universo territorial) · "
      f"${alto['val_contrato'].sum()/1e12:,.2f} billones")
alto["txt_banderas"].value_counts().head(10).rename("combinaciones de banderas más frecuentes")

Bandeja: 1,948 contratos (1.05 % del universo territorial) · $9.21 billones


txt_banderas
R1,R2,R5       644
R1,R3,R6       602
R1,R2,R6       279
R1,R2,R3       116
R1,R3,R5       110
R1,R3,R6,R8     49
R1,R2,R3,R6     42
R1,R2,R6,R8     19
R1,R2,R3,R8     16
R1,R5,R6        12
Name: combinaciones de banderas más frecuentes, dtype: int64